In [1]:
import json
from openai import OpenAI
from tqdm import tqdm
from datasets import load_dataset

/srv/nlprx-lab/share6/gramesh31/LLM-ambiguity/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import login
login(token="")

In [26]:
ds = load_dataset('./AbstentionBench', trust_remote_code=True)

Generating gpqa_abstain split: 80 examples [00:00, 195.06 examples/s]
Generating gsm8k_abstain split: 2426 examples [00:00, 7011.09 examples/s]
Generating mmlu_math_abstain split: 266 examples [00:00, 3523.31 examples/s]


In [30]:
# get number of examples with should_abstain = true
num_abstain = sum(1 for example in ds if example['should_abstain'])
print(f'Number of examples where should_abstain is true: {num_abstain}')

Number of examples where should_abstain is true: 1386


In [34]:
ds[-1]

{'question': 'In a non-leap year, what is the expected number of times Bob will roll his die?\nA. \\frac{5}{8}\nB. \\frac{1825}{4}\nC. \\frac{5}{4}\nD. 438',
 'reference_answers': ['D'],
 'should_abstain': True,
 'metadata_json': '{"subject": "high_school_mathematics"}'}

In [2]:
ds = load_dataset('newfacade/LeetCodeDataset', trust_remote_code=True)

In [3]:
# get all the hard problems
hard_problems = [example for example in ds['train'] if example['difficulty'] == 'Hard']
print(f'Number of hard problems: {len(hard_problems)}')

Number of hard problems: 606


In [34]:
hard_problems[0]['problem_description']

'Given two sorted arrays nums1 and nums2 of size m and n respectively, return the median of the two sorted arrays.\nThe overall run time complexity should be O(log (m+n)).\n\xa0\nExample 1:\n\nInput: nums1 = [1,3], nums2 = [2]\nOutput: 2.00000\nExplanation: merged array = [1,2,3] and median is 2.\n\nExample 2:\n\nInput: nums1 = [1,2], nums2 = [3,4]\nOutput: 2.50000\nExplanation: merged array = [1,2,3,4] and median is (2 + 3) / 2 = 2.5.\n\n\xa0\nConstraints:\n\nnums1.length == m\nnums2.length == n\n0 <= m <= 1000\n0 <= n <= 1000\n1 <= m + n <= 2000\n-106 <= nums1[i], nums2[i] <= 106\n\n'

In [39]:
problems = [(p['question_id'], p['problem_description'].split('Example 1')[0]) for p in hard_problems]

In [ ]:
client = OpenAI(api_key="")

In [ ]:
# dataset = load_dataset("openai/openai_humaneval", split="test[:{}]".format(5))
# prompts = [ex["prompt"] for ex in dataset]
# dataset = load_dataset("HuggingFaceH4/aime_2024", split="train")
# prompts = [ex["problem"] for ex in dataset]
#swe-bench


In [20]:
# load dataset APPS_dataset.jsonl
with open("APPS_dataset.jsonl", "r") as f:
    prompts = [json.loads(line)["question"] for line in f.readlines()]

In [21]:
prompts[0]

'Polycarp has $n$ different binary words. A word called binary if it contains only characters \'0\' and \'1\'. For example, these words are binary: "0001", "11", "0" and "0011100".\n\nPolycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words and each next word (starting from the second) must start with the last character of the previous word. The first word can be any. For example, these sequence of words can be named during the game: "0101", "1", "10", "00", "00001".\n\nWord reversal is the operation of reversing the order of the characters. For example, the word "0111" after the reversal becomes "1110", the word "11010" after the reversal becomes "01011".\n\nProbably, Polycarp has such a set of words that there is no way to put them in the order correspondent to the game rules. In this situation, he wants to reverse some words from his set so that:  the final set of $n$ words still contains different words (i.e. all words are unique)

In [4]:
print(prompts[0])

Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, the walk takes her 4 hours, including $t$ minutes spent in the coffee shop. When she walks $s+2$ kilometers per hour, the walk takes her 2 hours and 24 minutes, including $t$ minutes spent in the coffee shop. Suppose Aya walks at $s+\frac{1}{2}$ kilometers per hour. Find the number of minutes the walk takes her, including the $t$ minutes spent in the coffee shop.


In [7]:
SYSTEM_PROMPT = "You are a research assistant working on a project investigating how large language models (LLMs) process ambiguity in prompts they are given. The prompts being evaluated are complex, therefore you are not just searching for an ambiguous word; rather, you are trying to see what area of the prompt causes the underlying task asked in the prompt to be unclear to the LLM. You are currently working on producing a dataset for the project by generating ambiguous versions of given prompts."

In [36]:
USER_TEMPLATE = """Your task is to modify prompts such that LLMs would not have a clear understanding of the underlying task in the prompt. The obfuscation should be constrained to one contiguous section in the original prompt, not just a single word. If we were to locate the obscurity to one part of the prompt, it should obviously be the section that you modified. The section modified can be as long or short as needed as long as it results in the following behavior: The resulting prompt should still have the same underlying task as the original prompt, but it should be unclear to an LLM what exactly that task is to where it either cannot answer correctly, or will ask for more clarification.

Here is the clear prompt:

"{prompt}"


Return the modified prompt, the section in the original prompt that was modified, and a brief decription of the change made.
"""

In [50]:
USER_TEMPLATE = """Given the following coding prompt, your task is to first identify the top 3 sentences in the prompt are most important to the clarity of the problem. That is, you need to identify which sentences, if they were obscured or deleted, would cause the underlying task of the problem to be unclear. Then, you will construct an obscured version of each sentence in a manner that substituting the obscured version of the sentence into the original prompt would make the problem unclear. 

You must output the top 3 sentences meeting the above criteria in this schema exactly, without any additional response: 
{{
    original sentence 1: obscured sentence 1,
    original sentence 2: obscured sentence 2,
    original sentence 3: obscured sentence 3,
}}

original sentence 1 should be the sentence that is most critical to the clarity of the problem. 

Here is the coding prompt: 
"{prompt}"
"""

In [9]:
USER_TEMPLATE = """You are given a coding prompt consisting of multiple sentences.

Your task has two steps:

1. Identify the **three individual sentences** in the prompt that are most critical to understanding what the programmer is expected to implement. These should be the sentences whose removal or corruption would most severely impair comprehension of the task.

2. For each of the three selected sentences, produce an **obscured version** that preserves grammatical structure but removes or corrupts the key technical meaning such that, if substituted back into the prompt, the overall task would become unclear or misleading.

Rank the sentences by importance to task clarity:
- original sentence 1 must be the single most critical sentence,
- followed by original sentence 2,
- then original sentence 3.

You must output **only** the following JSON-like object, with no extra text before or after:

{{
    original sentence 1: obscured sentence 1,
    original sentence 2: obscured sentence 2,
    original sentence 3: obscured sentence 3
}}

Important constraints:
- Each key must be an exact sentence copied verbatim from the prompt.
- Each value must be an obscured version of that same sentence.
- Do not paraphrase the original sentences in the keys.
- Do not include explanations, bullet points, or formatting outside the schema.

Here is the coding prompt:
"{prompt}"
"""


In [28]:
USER_TEMPLATE = """You are given a coding prompt consisting of multiple sentences.

Your task has two steps:

1. Identify the **contiguous portion of text** in the prompt that is most critical to understanding what the programmer is expected to implement. This should be the phrase or sentences whose removal or corruption would most severely impair comprehension of the task.

2. For this portion of text, produce an **obscured version** that obfuscates the underlying technical meaning such that, if substituted back into the prompt, the overall task would become unclear. You must obscure the sentence enough that the coding problem is no longer understandable logically if substituted in. An LLM should not respond to the modified prompt because of how unclear it is. 
Try not to only use scope ambiguous vocabulary/quantifiers (like "maybe", "sometimes", "vaguely" or "random") for the obfuscation. Instead, consider using techniques like adding random information, deleting key information, and replacing words. Just make sure the meaning is obfuscated enough that the whole prompt becomes unclear and unanswerable.

You must output **only** the following JSON-like object, with no extra text before or after:

{{
   **contiguous portion of text**: **obscured version**,
}}

Important constraints:
- The key must be an exact contiguous portion of text copied verbatim from the prompt.
- The value must be an obscured version of that same text.
- Do not paraphrase the original text in the keys.
- Do not include explanations, bullet points, or formatting outside the schema.

Here is the coding prompt:
"{prompt}"
"""

In [24]:
SYSTEM_PROMPT = "You are a research assistant working on a project investigating how large language models (LLMs) process ambiguity in prompts they are given. The prompts being evaluated are complex, therefore you are trying to see what area of the prompt causes the underlying task asked in the prompt to be unclear to the LLM. You are currently working on producing a dataset for the project by generating ambiguous versions of given prompts."

In [40]:
def generate_unclear_examples(prompt_text):
    response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(prompt=prompt_text)},
            ],
            temperature=0.7,
            max_tokens=500,
        )
    content = response.choices[0].message.content.strip()
    return content

In [50]:
print(problems[300][1])

An undirected graph of n nodes is defined by edgeList, where edgeList[i] = [ui, vi, disi] denotes an edge between nodes ui and vi with distance disi. Note that there may be multiple edges between two nodes.
Given an array queries, where queries[j] = [pj, qj, limitj], your task is to determine for each queries[j] whether there is a path between pj and qj such that each edge on the path has a distance strictly less than limitj .
Return a boolean array answer, where answer.length == queries.length and the jth value of answer is true if there is a path for queries[j] is true, and false otherwise.
 



In [51]:
prompt = problems[300][1]
unclear_prompt = generate_unclear_examples(prompt)


In [52]:
print(unclear_prompt)

```json
{
   "your task is to determine for each queries[j] whether there is a path between pj and qj such that each edge on the path has a distance strictly less than limitj .": "your mission involves figuring out if each j element in queries has some sort of connection between pj and qj, where the connection might involve edges but no specific details about their measurement are given."
}
```


In [53]:
for i, (id,prompt) in enumerate(tqdm(problems)):
    if len(prompt.split('.')) < 5:
        continue
    unclear_data = generate_unclear_examples(prompt)
    if "```json" in unclear_data:
        unclear_data = unclear_data.split("```json")[1].split("```")[0].strip()

    with open(f'leetcode/unclear_portion/{id}.txt', 'w', encoding='utf-8') as f:
        f.write(unclear_data)

100%|██████████| 606/606 [11:37<00:00,  1.15s/it]


In [14]:
prompt = """A magnetic field is directed perpendicular to the plane of a circular coil of area 0.2 m^2 and 250 turns. If the magnetic field is increased from 0.01 T to 0.06 T during a time interval of 0.25 s, the average induced EMF in the coil is"""

In [19]:
repr(prompt)

"'Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, the walk takes her 4 hours, including $t$ minutes spent in the coffee shop. When she walks $s+2$ kilometers per hour, the walk takes her 2 hours and 24 minutes, including $t$ minutes spent in the coffee shop. Suppose Aya walks at $s+\\\\frac{1}{2}$ kilometers per hour. Find the number of minutes the walk takes her, including the $t$ minutes spent in the coffee shop.'"

In [27]:
prompt = """Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words and each next word (starting from the second) must start with the last character of the previous word. The first word can be any. For example, these sequence of words can be named during the game: "0101", "1", "10", "00", "00001".

Word reversal is the operation of reversing the order of the characters. For example, the word "0111" after the reversal becomes "1110", the word "11010" after the reversal becomes "01011".

Probably, Polycarp has such a set of words that there is no way to put them in the order correspondent to the game rules. In this situation, he wants to reverse some words from his set so that:  the final set of $n$ words still contains different words (i.e. all words are unique);  there is a way to put all words of the final set of words in the order so that the final sequence of $n$ words is consistent with the game rules. 

Polycarp wants to reverse minimal number of words."""

In [38]:
# prompt = prompts[0]
unclear_prompt = generate_unclear_examples(prompt)
print("Original Prompt:\n", prompt)
print("\nUnderspecified Prompt:\n", unclear_prompt)

Original Prompt:
 Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words and each next word (starting from the second) must start with the last character of the previous word. The first word can be any. For example, these sequence of words can be named during the game: "0101", "1", "10", "00", "00001".

Word reversal is the operation of reversing the order of the characters. For example, the word "0111" after the reversal becomes "1110", the word "11010" after the reversal becomes "01011".

Probably, Polycarp has such a set of words that there is no way to put them in the order correspondent to the game rules. In this situation, he wants to reverse some words from his set so that:  the final set of $n$ words still contains different words (i.e. all words are 

In [22]:
generate_unclear_examples(prompt)

'"A magnetic field is directed perpendicular to the plane of a circular coil with 250 turns. If the magnetic field is increased from 0.01 T to 0.06 T during a time interval of 0.25 s, the average induced EMF in the coil is"'